# LLM-engineering ladder ablation

Evaluates one model at every rung of the ladder **prompt → context → harness → loop → graph**, so the
contribution of each stage (and of GRPO on top of the best non-RL setup) can be measured separately.

**Protocol**
- **Time splits:** train 2016–2019 (GRPO), **dev** 2020–2021 (choosing the best option per stage), **test** 2022–2024 (reported numbers only).
- **Greedy stage-wise selection on dev:** evaluate every variant of a stage on top of the best config so far, then keep the best (`SELECT_METRIC`).
- **No leakage:** harness/loop feedback (errors, in-sample metrics) comes only from the 2 years *before* the evaluated window. Market-summary context uses the same pre-window data. The evaluated window is backtested once, on the final strategy.
- **Metrics:** rates over *all* samples (a crash counts as a failure) with Wilson 95% CIs; returns with bootstrap CIs; buy-and-hold excess; paired, symbol-level bootstrap tests between rungs.
- **RL rung:** train GRPO with the *selected* prompt + context (`prompt_fn(best)`), then re-run this notebook with `LORA_ADAPTER` set and `SELECTION_FROM` pointing at the base run's `selection.json`.

Runs are resumable: results are appended to CSVs in `OUT_DIR`, and finished (config, symbol, sample) rows are skipped.

In [ ]:
%pip install -q matplotlib pandas requests

# Shared backtest / prompt / reward code: the `trading_rl` package in this repo
import os, sys
if os.path.isdir('../trading_rl'):   # local clone: import straight from the repo
    sys.path.insert(0, os.path.abspath('..'))
else:                                # Colab / Kaggle
    %pip install -q "trading-rl @ git+https://github.com/adhamhelmy/llm-fine-tuning.git"

### Config

In [ ]:
# --- Model backend: 'ollama' (local), 'mlx' (Apple Silicon) or 'unsloth' (CUDA / Colab) ---
BACKEND        = 'ollama'
MODEL          = 'qwen3.8:27b-mlx'      # ollama tag | MLX path/repo | HF model id
LORA_ADAPTER   = None                   # GRPO adapter (MLX-converted for 'mlx', PEFT for 'unsloth')
OLLAMA_URL     = 'http://localhost:11434'
MAX_NEW_TOKENS = 1536
TEMPERATURE    = 1.0

# --- Secrets (fill in at runtime) ---
HF_TOKEN=''
ALPACA_API_KEY=''
ALPACA_SECRET_KEY=''

# --- Experiment ---
from trading_rl import TESTER_SYMBOLS, DEV_START, DEV_END, TEST_START, TEST_END
SYMBOLS        = TESTER_SYMBOLS[:5]                                                                                                               
N_SAMPLES_DEV  = 1                                                                                                                                
N_SAMPLES_TEST = 2
STAGES_TO_RUN  = ['prompt', 'context', 'harness', 'loop', 'graph']
SELECT_METRIC  = 'profitable'     # or 'beat_bh', 'mean_reward', 'traded'
SELECTION_FROM = None             # path to another run's selection.json: skip dev selection
TIMEOUT_SECONDS = 10

MODEL_LABEL = MODEL.split('/')[-1].replace(':', '-') + ('+lora' if LORA_ADAPTER else '')
OUT_DIR     = f'ladder_results/{MODEL_LABEL}'

### Imports

In [ ]:
import json
import os

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from trading_rl import (Backtester, LadderConfig, baseline_table, dump_prompts, format_report,
                        paired_delta, report, run_ablation, select_best, stage_configs)
from trading_rl.ladder import feedback_window, markdown_table

os.makedirs(OUT_DIR, exist_ok=True)

### Model backend
Each backend exposes `generate(prompt) -> str`. The ladder code is backend-agnostic.

In [ ]:
if BACKEND == 'ollama':
    import requests
    tags = [m['name'] for m in requests.get(f'{OLLAMA_URL}/api/tags').json().get('models', [])]
    assert any(MODEL in t for t in tags), f'Run: ollama pull {MODEL}'

    def generate(prompt):
        r = requests.post(f'{OLLAMA_URL}/api/chat', timeout=600, json={
            'model': MODEL, 'stream': False,
            'messages': [{'role': 'user', 'content': prompt}],
            'options': {'temperature': TEMPERATURE, 'num_predict': MAX_NEW_TOKENS, 'num_ctx': 8192}})
        r.raise_for_status()
        return r.json()['message']['content']

elif BACKEND == 'mlx':
    from mlx_lm import generate as mlx_generate, load as mlx_load
    from mlx_lm.sample_utils import make_sampler
    mlx_model, mlx_tok = mlx_load(MODEL, adapter_path=LORA_ADAPTER)
    sampler = make_sampler(temp=TEMPERATURE)

    def generate(prompt):
        text = mlx_tok.apply_chat_template([{'role': 'user', 'content': prompt}],
                                           tokenize=False, add_generation_prompt=True)
        return mlx_generate(mlx_model, mlx_tok, prompt=text, max_tokens=MAX_NEW_TOKENS,
                            sampler=sampler, verbose=False)

elif BACKEND == 'unsloth':
    from trading_rl.model import Unsloth   # import before any trl/transformers import
    llm = Unsloth(MODEL, max_seq_length=8192, lora_adapter_path=LORA_ADAPTER,
                  adapter_trainable=False, hf_token=HF_TOKEN)

    def generate(prompt):
        return llm.generate(prompt, temperature=TEMPERATURE, max_new_tokens=MAX_NEW_TOKENS)

else:
    raise ValueError(BACKEND)

### Data and buy-and-hold baselines
Bars for the evaluation windows *and* the feedback windows before them.

In [ ]:
backtester = Backtester(ALPACA_API_KEY, ALPACA_SECRET_KEY, verbose=False)
for start, end in [(DEV_START, DEV_END), (TEST_START, TEST_END),
                   feedback_window(DEV_START), feedback_window(TEST_START)]:
    backtester.load_bars(SYMBOLS, start, end)

baselines = pd.concat([baseline_table(backtester, SYMBOLS, DEV_START, DEV_END),
                       baseline_table(backtester, SYMBOLS, TEST_START, TEST_END)])
baselines.to_csv(f'{OUT_DIR}/baselines.csv', index=False)
display(report(baselines, baselines).pipe(format_report))

### Stage-wise selection on the dev window
For each stage, every variant is built on top of the best config so far. The table shows each variant with CIs,
and the paired test compares each variant against the stage's starting point.

In [ ]:
dev_csv = f'{OUT_DIR}/dev.csv'
if SELECTION_FROM:
    selection = json.load(open(SELECTION_FROM))
    print(f'Using selection from {SELECTION_FROM}')
else:
    best, selection = LadderConfig(), {'base': LadderConfig().to_json()}
    for stage in STAGES_TO_RUN:
        configs = stage_configs(stage, best)
        print(f"\n{'#'*70}\n# Stage: {stage}  (base: {best.name})\n{'#'*70}")
        df_dev = run_ablation(generate, backtester, configs, SYMBOLS, DEV_START, DEV_END,
                              N_SAMPLES_DEV, dev_csv, model=MODEL_LABEL, timeout=TIMEOUT_SECONDS)
        rep = report(df_dev, baselines)
        display(format_report(rep))
        display(pd.DataFrame([paired_delta(df_dev, best.name, c.name, SELECT_METRIC, baselines)
                              for c in configs if c != best]))
        best = select_best(rep, configs, SELECT_METRIC)
        selection[stage] = best.to_json()
        print(f'==> best after {stage}: {best.name}')
    json.dump(selection, open(f'{OUT_DIR}/selection.json', 'w'), indent=2)

rungs = {k: LadderConfig(**json.loads(v)) for k, v in selection.items()}
for k, c in rungs.items():
    print(f'{k:8} {c.name}')

### Final evaluation on the held-out test window
One row per rung: the base prompt, then the selected config after each stage. The memorization probe hides the
ticker and dates from the final config: a large drop would suggest the model relies on remembered price history.

In [ ]:
test_configs = list(dict.fromkeys(rungs.values()))
final = test_configs[-1]
probe = final.replace(anonymize=True)
test_csv = f'{OUT_DIR}/test.csv'
df_test = run_ablation(generate, backtester, test_configs + [probe], SYMBOLS, TEST_START, TEST_END,
                       N_SAMPLES_TEST, test_csv, model=MODEL_LABEL, timeout=TIMEOUT_SECONDS)

rep_test = report(df_test, baselines)
rep_test.insert(1, 'rung', rep_test['config'].map(
    {**{c.name: k for k, c in reversed(list(rungs.items()))}, probe.name: 'probe'}))
display(format_report(rep_test).assign(rung=rep_test['rung']))

steps = [paired_delta(df_test, a.name, b.name, m, baselines)
         for a, b in zip(test_configs, test_configs[1:]) for m in ('valid', 'profitable', 'beat_bh')]
steps.append(paired_delta(df_test, final.name, probe.name, 'profitable', baselines))
deltas = pd.DataFrame(steps)
display(deltas)

### Export for the paper

In [ ]:
rep_test.to_csv(f'{OUT_DIR}/test_report.csv', index=False)
open(f'{OUT_DIR}/test_report.md', 'w').write(markdown_table(format_report(rep_test).assign(rung=rep_test['rung'])))
deltas.to_csv(f'{OUT_DIR}/test_deltas.csv', index=False)
dump_prompts(f'{OUT_DIR}/prompts', test_configs + [probe], backtester, symbol=SYMBOLS[0])

fig, ax = plt.subplots(figsize=(10, 4))
for col, label in [('valid', 'runs without error'), ('profitable', 'profitable'), ('beat_bh', 'beats buy-and-hold')]:
    y = rep_test[col] * 100
    err = [(y - rep_test[f'{col}_lo'] * 100).clip(lower=0), (rep_test[f'{col}_hi'] * 100 - y).clip(lower=0)]
    ax.errorbar(rep_test['rung'], y, yerr=err,
                marker='o', capsize=4, label=label)
ax.set_ylabel('% of all samples (95% CI)')
ax.set_title(f'{MODEL_LABEL}: test {TEST_START} to {TEST_END}')
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/ladder.png', dpi=160)
plt.show()

### Next: the RL rung
1. Train GRPO with the selected prompt + context (harness/loop are inference-time and not part of the training prompt):
   ```python
   from trading_rl import prompt_fn, TRAIN_START, TRAIN_END, TRAINING_SYMBOLS
   from trading_rl.dataset import build_dataset
   dataset = build_dataset(TRAINING_SYMBOLS, TRAIN_START, TRAIN_END, prompt_fn=prompt_fn(rungs['context'], backtester))
   ```
   Set `max_prompt_length` from the tokenized prompt, which is much longer with context.
2. Re-run this notebook with `LORA_ADAPTER=<adapter>` and `SELECTION_FROM=f'ladder_results/{base label}/selection.json'`.
3. Compare base and RL models at the *same* rung: concatenate both `test.csv` files, keep one `config`, then
   `paired_delta(df, base_label, rl_label, 'profitable', baselines, by='model')`.